In [1]:
#!/usr/bin/env python3
"""
Titanic survival prediction
===========================
Combines two sources of information, both treated as important:

  A) DATASET features  - real columns from the Kaggle file
                         (Pclass, Sex, Age, SibSp, Parch, Fare, Cabin deck, Embarked, Title ...)
  B) SCENARIO features - your 6 extra points about the sinking:
        1. distance to the lifeboat
        2. is the path to the boat clear
        3. is the person's age suited to reacting quickly
        4. has an emergency life jacket
        5. how much rush / crowding there is at the boats
        6. how many fit people (by age range) are able to help others

IMPORTANT - about points 1, 2, 4, 5:
The Kaggle file has NO columns for them. They are therefore built as *simulated
scenario features* from real passenger info (deck, class, family size, age) plus
random noise. They are NEVER computed from the Survived column, so there is no
label leakage. Points 3 and 6 are derived from the real Age / Ticket columns.
Because the simulated points are derived from the same columns, they cannot add
brand-new real-world information; the script reports Dataset-only vs
Scenario-only vs Combined so you can see exactly what they contribute.

Usage:
    python titanic_survival.py                    # downloads data with kagglehub
    python titanic_survival.py --csv Titanic-Dataset.csv
"""
import argparse
import glob
import os
import sys

import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
OUT_DIR = "titanic_output"

# --------------------------------------------------------------------------- #
# 1. Load
# --------------------------------------------------------------------------- #
def find_local_csv():
    """Look for the Titanic CSV next to this script (or in ./data) before downloading."""
    # __file__ does not exist inside a Colab / Jupyter cell, so fall back to the working folder
    here = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
    for folder in (here, os.path.join(here, "data"), os.getcwd()):
        for name in ("Titanic-Dataset.csv", "titanic.csv", "train.csv"):
            p = os.path.join(folder, name)
            if os.path.isfile(p):
                return p
    return None


def load_data(csv_path=None):
    if csv_path is None:
        csv_path = find_local_csv()
    if csv_path is None:
        import kagglehub

        folder = kagglehub.dataset_download("yasserh/titanic-dataset")
        print("Path to dataset files:", folder)
        files = sorted(glob.glob(os.path.join(folder, "*.csv")))
        if not files:
            sys.exit(f"No CSV found in {folder}")
        csv_path = files[0]
    df = pd.read_csv(csv_path)
    need = {"Survived", "Pclass", "Name", "Sex", "Age", "SibSp", "Parch",
            "Ticket", "Fare", "Cabin", "Embarked"}
    missing = need - set(df.columns)
    if missing:
        sys.exit(f"Dataset is missing columns: {sorted(missing)}")
    print(f"Loaded {len(df)} passengers from {csv_path}")
    return df


# --------------------------------------------------------------------------- #
# 2. Dataset features (real columns, cleaned)
# --------------------------------------------------------------------------- #
def add_dataset_features(df):
    df = df.copy()
    title = df["Name"].str.extract(r",\s*([^\.]+)\.", expand=False).str.strip()
    title = title.replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
    common = {"Mr", "Mrs", "Miss", "Master"}
    df["Title"] = title.where(title.isin(common), "Rare")

    # Age: impute from title group median (uses no target information)
    df["AgeWasMissing"] = df["Age"].isna().astype(int)
    df["Age"] = df["Age"].fillna(df.groupby("Title")["Age"].transform("median"))
    df["Age"] = df["Age"].fillna(df["Age"].median())

    df["Fare"] = df["Fare"].fillna(df["Fare"].median())
    df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode().iloc[0])
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)
    df["Deck"] = df["Cabin"].astype(str).str[0].where(df["Cabin"].notna(), "U")
    return df


# --------------------------------------------------------------------------- #
# 3. Scenario features (your 6 points)
# --------------------------------------------------------------------------- #
DECK_DISTANCE_RANK = {"T": 1, "A": 1, "B": 2, "C": 3, "D": 4, "E": 5, "F": 6, "G": 7}
CLASS_DEFAULT_RANK = {1: 2.5, 2: 5.0, 3: 6.0}  # used when Cabin/deck is unknown


def age_fitness(age):
    """Point 6: physical fitness (0-1) by age range. Age only, no sex assumption."""
    return np.select(
        [age < 5, age < 12, age < 16, age <= 35, age <= 50, age <= 60],
        [0.10, 0.25, 0.60, 1.00, 0.85, 0.55],
        default=0.30,
    )


def add_scenario_features(df, seed=SEED):
    """Never touches df['Survived']."""
    rng = np.random.default_rng(seed)
    n = len(df)
    df = df.copy()

    # ---- 5. boat rush / crowding (0 calm .. 1 stampede) --------------------
    base_rush = df["Pclass"].map({1: 0.40, 2: 0.60, 3: 0.85})
    df["rush_index"] = (base_rush + 0.04 * (df["FamilySize"] - 1)
                        + rng.normal(0, 0.10, n)).clip(0, 1)

    # ---- 1. distance to lifeboat (0 = next to boat .. 100 = far) ----------
    rank = df["Deck"].map(DECK_DISTANCE_RANK)
    rank = rank.fillna(df["Pclass"].map(CLASS_DEFAULT_RANK))
    df["distance_index"] = (rank * 12 + rng.normal(0, 8, n)).clip(0, 100)

    # ---- 2. path clear? (1 = clear) ---------------------------------------
    p_clear = (df["Pclass"].map({1: 0.85, 2: 0.65, 3: 0.45})
               - 0.03 * (rank - 1) - 0.25 * df["rush_index"]).clip(0.05, 0.95)
    df["path_clear"] = (rng.random(n) < p_clear).astype(int)

    # ---- 3. age -> ability to react quickly (0-1) -------------------------
    age = df["Age"]
    reaction = np.select(
        [age < 5, age < 12, age < 18, age <= 45, age <= 60],
        [0.20, 0.50, 0.80, 1.00, 0.75],
        default=0.40,
    )
    df["reaction_score"] = (reaction + rng.normal(0, 0.05, n)).clip(0, 1)

    # ---- 4. emergency life jacket (1 = has one) ---------------------------
    p_jacket = df["Pclass"].map({1: 0.97, 2: 0.95, 3: 0.90})
    df["has_lifejacket"] = (rng.random(n) < p_jacket).astype(int)

    # ---- 6. fitness + capable of saving others ----------------------------
    df["fitness_score"] = age_fitness(age.to_numpy())
    df["capable_rescuer"] = ((df["fitness_score"] >= 0.70) & (age >= 16)).astype(int)
    df["is_dependent"] = ((age < 12) | (age > 60)).astype(int)

    # People sharing a Ticket travelled together: count helpers around each person
    grp = df.groupby("Ticket")
    df["group_rescuers"] = grp["capable_rescuer"].transform("sum") - df["capable_rescuer"]
    df["group_dependents"] = grp["is_dependent"].transform("sum") - df["is_dependent"]
    df["rescuer_per_dependent"] = df["group_rescuers"] / (df["group_dependents"] + 1)
    return df


def fitness_by_age_range(df):
    """Point 6 headline answer: how many people are fit / able to save others, per age range."""
    bins = [0, 12, 16, 25, 35, 50, 60, 120]
    labels = ["0-11", "12-15", "16-24", "25-34", "35-49", "50-59", "60+"]
    t = df.assign(age_range=pd.cut(df["Age"], bins=bins, labels=labels, right=False))
    # "fit" = fitness >= 0.50 (moderately fit or better)
    # "capable of saving others" = fitness >= 0.70 and adult (16+), a stricter bar
    t = t.assign(is_fit=(t["fitness_score"] >= 0.50).astype(int))

    def summarise(g):
        return pd.Series({
            "passengers": len(g),
            "fit_people": int(g["is_fit"].sum()),
            "capable_of_saving_others": int(g["capable_rescuer"].sum()),
            "avg_fitness": round(g["fitness_score"].mean(), 2),
            "pct_capable": round(100 * g["capable_rescuer"].mean(), 1),
        })

    out = t.groupby("age_range", observed=True).apply(summarise)
    out.loc["TOTAL"] = summarise(t)
    count_cols = ["passengers", "fit_people", "capable_of_saving_others"]
    out[count_cols] = out[count_cols].astype(int)
    return out


# --------------------------------------------------------------------------- #
# 4. Models
# --------------------------------------------------------------------------- #
DATASET_NUM = ["Pclass", "Age", "SibSp", "Parch", "Fare", "FamilySize", "IsAlone", "AgeWasMissing"]
DATASET_CAT = ["Sex", "Embarked", "Deck", "Title"]
SCENARIO_NUM = ["distance_index", "path_clear", "reaction_score", "has_lifejacket",
                "rush_index", "fitness_score", "capable_rescuer",
                "group_rescuers", "group_dependents", "rescuer_per_dependent"]

FEATURE_SETS = {
    "Dataset only": (DATASET_NUM, DATASET_CAT),
    "Scenario only (points 1-6)": (SCENARIO_NUM, []),
    "Dataset + Scenario": (DATASET_NUM + SCENARIO_NUM, DATASET_CAT),
}

POINT_OF = {
    "distance_index": "1 Distance to boat", "path_clear": "2 Clear path",
    "reaction_score": "3 Age / reaction", "has_lifejacket": "4 Life jacket",
    "rush_index": "5 Boat rush", "fitness_score": "6 Fitness / rescuers",
    "capable_rescuer": "6 Fitness / rescuers", "group_rescuers": "6 Fitness / rescuers",
    "group_dependents": "6 Fitness / rescuers", "rescuer_per_dependent": "6 Fitness / rescuers",
}


def make_pipeline(model, num, cat):
    transformers = [("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                                      ("sc", StandardScaler())]), num)]
    if cat:
        transformers.append(("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat))
    return Pipeline([("prep", ColumnTransformer(transformers)), ("clf", model)])


def get_models():
    return {
        "Logistic Regression": LogisticRegression(max_iter=2000, random_state=SEED),
        "Random Forest": RandomForestClassifier(n_estimators=400, min_samples_leaf=2,
                                                random_state=SEED, n_jobs=-1),
        "Gradient Boosting": GradientBoostingClassifier(random_state=SEED),
    }


def evaluate(df):
    y = df["Survived"].astype(int)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]
    rows = []
    for fs_name, (num, cat) in FEATURE_SETS.items():
        X = df[num + cat]
        for m_name, model in get_models().items():
            res = cross_validate(make_pipeline(model, num, cat), X, y, cv=cv, scoring=scoring)
            rows.append({"features": fs_name, "model": m_name,
                         **{k: res[f"test_{k}"].mean() for k in scoring},
                         "accuracy_std": res["test_accuracy"].std()})
    return pd.DataFrame(rows).round(4)


def importance_report(df):
    num, cat = FEATURE_SETS["Dataset + Scenario"]
    pipe = make_pipeline(get_models()["Random Forest"], num, cat)
    pipe.fit(df[num + cat], df["Survived"].astype(int))
    names = pipe.named_steps["prep"].get_feature_names_out()
    imp = pd.Series(pipe.named_steps["clf"].feature_importances_, index=names)

    def base(n):
        n = n.split("__", 1)[1]
        for c in cat:
            if n.startswith(c + "_"):
                return c
        return n

    per_feature = imp.groupby(base).sum().sort_values(ascending=False)
    per_point = per_feature.groupby(lambda f: POINT_OF.get(f, "Dataset columns")).sum()
    return per_feature, per_point.sort_values(ascending=False)


# --------------------------------------------------------------------------- #
# 5. Main
# --------------------------------------------------------------------------- #
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--csv", help="path to Titanic CSV (skips the kagglehub download)")
    ap.add_argument("--out", default=OUT_DIR)
    args, _ = ap.parse_known_args()  # ignores extra arguments Jupyter/Colab passes in
    os.makedirs(args.out, exist_ok=True)

    df = load_data(args.csv)
    df = add_scenario_features(add_dataset_features(df))

    # Safety check: scenario features must not be derived from the label
    corr_cols = ["path_clear", "has_lifejacket", "rush_index", "distance_index"]
    print("\nCorrelation of simulated features with Survived (sanity check):")
    print(df[corr_cols + ["Survived"]].corr()["Survived"].drop("Survived").round(3).to_string())

    # Point 6
    fit_table = fitness_by_age_range(df)
    fit_table.to_csv(os.path.join(args.out, "fitness_by_age_range.csv"))
    print("\nPoint 6 - fit people / capable of saving others, by age range:")
    print(fit_table.to_string())

    # Models
    results = evaluate(df)
    results.to_csv(os.path.join(args.out, "model_results.csv"), index=False)
    print("\nCross-validated results (5-fold):")
    print(results.to_string(index=False))

    # Best combined model: confusion matrix
    comb = results[results["features"] == "Dataset + Scenario"].sort_values("accuracy", ascending=False)
    best_name = comb.iloc[0]["model"]
    num, cat = FEATURE_SETS["Dataset + Scenario"]
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    pred = cross_val_predict(make_pipeline(get_models()[best_name], num, cat),
                             df[num + cat], df["Survived"].astype(int), cv=cv)
    cm = confusion_matrix(df["Survived"].astype(int), pred)
    fig, ax = plt.subplots(figsize=(4.5, 4))
    ConfusionMatrixDisplay(cm, display_labels=["Died", "Survived"]).plot(ax=ax, colorbar=False)
    ax.set_title(f"{best_name} (Dataset + Scenario)")
    fig.tight_layout()
    fig.savefig(os.path.join(args.out, "confusion_matrix.png"), dpi=150)
    plt.close(fig)

    # Importance mapped back to your points
    per_feature, per_point = importance_report(df)
    print("\nFeature importance grouped by your points (Random Forest):")
    print(per_point.round(3).to_string())
    print("\nTop 10 individual features:")
    print(per_feature.head(10).round(3).to_string())
    fig, ax = plt.subplots(figsize=(6, 3.5))
    per_point.sort_values().plot.barh(ax=ax)
    ax.set_xlabel("share of Random Forest importance")
    fig.tight_layout()
    fig.savefig(os.path.join(args.out, "importance_by_point.png"), dpi=150)
    plt.close(fig)

    df.to_csv(os.path.join(args.out, "titanic_with_features.csv"), index=False)
    print(f"\nSaved outputs to {os.path.abspath(args.out)}")


if __name__ == "__main__":
    main()

Using Colab cache for faster access to the 'titanic-dataset' dataset.
Path to dataset files: /kaggle/input/titanic-dataset
Loaded 891 passengers from /kaggle/input/titanic-dataset/Titanic-Dataset.csv

Correlation of simulated features with Survived (sanity check):
path_clear        0.218
has_lifejacket    0.027
rush_index       -0.277
distance_index   -0.267

Point 6 - fit people / capable of saving others, by age range:
           passengers  fit_people  capable_of_saving_others  avg_fitness  pct_capable
age_range                                                                            
0-11               72           0                         0         0.16          0.0
12-15              15          15                         0         0.60          0.0
16-24             231         231                       231         1.00        100.0
25-34             320         320                       320         1.00        100.0
35-49             179         179                       179

/tmp/ipykernel_1864/960994281.py:194: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  out = t.groupby("age_range", observed=True).apply(summarise)



Cross-validated results (5-fold):
                  features               model  accuracy  precision  recall     f1  roc_auc  accuracy_std
              Dataset only Logistic Regression    0.8305     0.7865  0.7659 0.7756   0.8737        0.0149
              Dataset only       Random Forest    0.8294     0.8139  0.7222 0.7644   0.8806        0.0138
              Dataset only   Gradient Boosting    0.8372     0.8277  0.7279 0.7742   0.8788        0.0207
Scenario only (points 1-6) Logistic Regression    0.6824     0.6292  0.4269 0.5079   0.6988        0.0194
Scenario only (points 1-6)       Random Forest    0.6823     0.6104  0.4826 0.5378   0.7066        0.0171
Scenario only (points 1-6)   Gradient Boosting    0.6980     0.6361  0.4943 0.5550   0.7132        0.0273
        Dataset + Scenario Logistic Regression    0.8249     0.7767  0.7630 0.7693   0.8727        0.0165
        Dataset + Scenario       Random Forest    0.8282     0.8134  0.7192 0.7628   0.8759        0.0223
        Dat